# Grammaires probabilistes et parsing CYK — de la CFG a la derivation la plus probable

**Navigation** : [<< Précédent — 03. CRF et étiquetage séquentiel](03_CRF_Etiquetage_Sequentiel.ipynb) | [Index de la série](README.md) | **04 (ce notebook)** | [Suivant — 05. HMM et Viterbi >>](05-HMM-Viterbi.ipynb)

**Sommaire** : [1. CFG explicite](#1-une-grammaire-a-contexte-libre-explicite) · [2. Productions et CNF](#2-productions-completes-et-forme-normale-de-chomsky) · [3. Recognizer CYK](#3-le-recognizer-cyk-programmation-dynamique-en-table-triangulaire) · [4. PCFG et treebank](#4-la-pcfg-estimer-les-probabilites-sur-un-mini-treebank) · [5. Viterbi et inside](#5-viterbi-et-inside-la-meilleure-derivation-nest-pas-la-phrase) · [6. Ambiguïté mesuree](#6-ambiguïté-mesuree-compter-les-derivations-et-peser-les-lectures) · [7. Témoin NLTK](#7-témoin-nltk-verifier-notre-parseur-contre-la-reference) · [8. Exercices](#8-exercices) · [Conclusion](#à-retenir)

La serie Texte traite le langage **cote modèle** : prompts, RAG, fine-tuning. Le notebook [`01_TAL_Du_Mot_Aux_Dependances.ipynb`](01_TAL_Du_Mot_Aux_Dependances.ipynb) a execute la seconde tradition — le **TAL classique** — au niveau du *mot* : lemmes, etiquettes morphosyntaxiques, dependances, entites nommees. Il manquait l'etage au-dessus : la **structure syntaxique** d'une phrase entiere.

C'est le rôle d'une **grammaire a contexte libre** (CFG) : un ensemble de règles de reecriture `S -> NP VP` qui decompose recursivement une phrase en constituants. Une CFG repond *oui* ou *non* a la question « cette phrase est-elle grammaticale ? ». Une **PCFG** — sa version ponderee par des probabilites — repond a la question plus fine : « *quelle* analyse syntaxique est la plus probable ? », ce qui importe des qu'une phrase est **ambigue**.

Ce notebook construit tout cela **from scratch**, sur un mini-francais fil rouge (un robot, un ballon, une boite) :

1. une CFG explicite, convertie en **forme normale de Chomsky (CNF)** par un convertisseur documente qui **preserve les probabilites** ;
2. le **recognizer CYK** — programmation dynamique en table triangulaire — visualise cellule par cellule ;
3. l'estimation des probabilites de règles par **maximum de vraisemblance** sur un mini-treebank construit et explique ici même ;
4. le decodage **Viterbi** de la meilleure derivation, avec backpointers et reconstruction de l'arbre dans la grammaire d'origine ;
5. l'**ambiguïté mesuree** : nombre de derivations, probabilites comparees, et la distinction nette entre *meilleure derivation* et *probabilite totale de la phrase* (somme de toutes les derivations, algorithme inside) ;
6. un **temoin** : le même calcul refait par `nltk`, la bibliotheque de reference, pour verifier que notre implementation donne *exactement* les mêmes nombres.

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Ecrire** une CFG et la convertir en CNF en preservant les probabilites ;
2. **Implementer** CYK (table triangulaire, complexite O(n^3 |G|)) et lire sa table ;
3. **Estimer** une PCFG par maximum de vraisemblance sur un treebank et mesurer son biais ;
4. **Decoder** la derivation la plus probable (Viterbi + backpointers) et la distinguer de la probabilite totale (inside) ;
5. **Verifier** une implementation de parseur contre un temoin de reference (`nltk`).

### Prerequis

- C#, CPython et `nltk` pour les témoins (CPU uniquement, aucun telechargement de corpus NLTK requis) ;
- notions du notebook [`01_TAL_Du_Mot_Aux_Dependances.ipynb`](01_TAL_Du_Mot_Aux_Dependances.ipynb) (catégories Det/N/V/P) recommandees.

### Duree estimee : 55 minutes

**Version jumelle** : [Python](04_PCFG_CYK_Parsing.ipynb). Avant Jupyter, installer CPython et NLTK et définir `PYTHONNET_PYDLL` vers la bibliothèque partagée ; `PYTHONHOME` et `PYTHONPATH` peuvent sélectionner l’environnement. Python.NET sert uniquement aux témoins NLTK.


## 1. Une grammaire a contexte libre explicite

Une **CFG** est un quadruplet : non-terminaux `N` (`S`, `NP`, `VP`, `PP`, `Det`, `N`, `V`, `P`), terminaux `T` (les mots), un axiome `S`, et des **règles de production** `A -> alpha` ou `A` est un non-terminal et `alpha` une sequence de symboles. Le « contexte libre » signifie que la recriture d'un non-terminal ne depend **pas** de son entourage — c'est ce qui rend le parsing polynomial possible.

Notre mini-francais : sept règles syntaxiques (phrase, groupe nominal avec ou sans complement prepositionnel, groupe verbal intransitif / transitif / transitif+complement, groupe prepositionnel) et un lexique borne de quatre catégories. Les deux règles `NP -> Det N PP` et `VP -> V NP PP` coexistent : c'est ce qui rendra la phrase fil rouge **ambigue** (le complement prepositionnel peut s'attacher au nom ou au verbe).


In [1]:
#r "nuget: pythonnet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Text.RegularExpressions;
using System.Globalization;
using System.Net;
using Microsoft.DotNet.Interactive.Formatting;
using Python.Runtime;
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((v,w)=>w.Write(v.Content),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);string E(string s)=>WebUtility.HtmlEncode(s);
record Rule(string Lhs,string Rhs) {public string[] Parts=>Rhs.Split(' ',StringSplitOptions.RemoveEmptyEntries);}
record Origin(string Kind,string Terminal,Rule[] Chain);
class Node {public string Sym,Word;public Rule Rule;public List<Node> Children;public Node(string sym,Rule rule=null,string word=null,IEnumerable<Node> children=null){Sym=sym;Rule=rule;Word=word;Children=children?.ToList()??new List<Node>();}}
var productions=new List<Rule> {
    new Rule("S","NP VP"),
    new Rule("NP","Det N"),
    new Rule("NP","Det N PP"),
    new Rule("VP","V"),
    new Rule("VP","V NP"),
    new Rule("VP","V NP PP"),
    new Rule("PP","P NP"),
    new Rule("Det","le"),
    new Rule("Det","la"),
    new Rule("Det","un"),
    new Rule("Det","une"),
    new Rule("N","robot"),
    new Rule("N","ballon"),
    new Rule("N","boite"),
    new Rule("N","camembert"),
    new Rule("N","couteau"),
    new Rule("N","table"),
    new Rule("V","voit"),
    new Rule("V","mange"),
    new Rule("V","trouve"),
    new Rule("V","coupe"),
    new Rule("V","dort"),
    new Rule("P","dans"),
    new Rule("P","sur"),
    new Rule("P","avec")
};
Console.WriteLine($".NET {Environment.Version} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version} | 7 règles syntaxiques + 18 lexicales = {productions.Count}");
foreach(var r in productions)Console.WriteLine($"{r.Lhs} → {r.Rhs}");

Installed Packages pythonnet, 3.0.5

.NET 10.0.14 | Python.NET 3.0.5.0 | 7 règles syntaxiques + 18 lexicales = 25


S → NP VP


NP → Det N


NP → Det N PP


VP → V


... (17 outputs left out)

V → dort


P → dans


P → sur


P → avec


### Lecture — la grammaire fil rouge : 25 productions, deux attachements

La grammaire declare **7 règles syntaxiques** et **18 règles lexicales** (4 determinants + 6 noms + 5 verbes + 3 prepositions), soit **25 productions**. La grammaire accepte des phrases de longueur arbitraire (les complements prepositionnels peuvent s'empiler), mais notre jeu de test les borne a trois a huit mots ; surtout, elle autorise les deux formes « avec PP » : `NP -> Det N PP` (le complement se rattache au *nom*) et `VP -> V NP PP` (le complement se rattache au *verbe*).

**Point cle** : cette grammaire reste volontairement minuscule. Tout ce qui suit — table CYK, probabilites, Viterbi — sera *lisible* précisément parce que chaque règle et chaque cellule de table tient dans l'ecran.


## 2. Productions completes et forme normale de Chomsky

CYK exige une grammaire en **forme normale de Chomsky** : chaque règle est soit `A -> B C` (deux non-terminaux), soit `A -> w` (un seul mot). Notre grammaire viole les deux formes : `VP -> V NP PP` a **trois** fils, `VP -> V` est une **règle unitaire** (un non-terminal seul), et aucun terminal n'apparait dans un rhs long ici mais le convertisseur doit savoir les traiter.

Le convertisseur `Convertir` applique trois transformations sur le domaine admis :

| Transformation | Règle d'origine | Règles CNF produites | Probabilite |
|---|---|---|---|
| **Terminal dans un rhs long** | `A -> B w` | `A -> B TERM_w`, `TERM_w -> w` | `p` garde par `A`, `TERM_w -> w` a 1 |
| **Binarisation** | `A -> B C D` | `A -> B M`, `M -> C D` | `p` sur la tete, 1 sur la suite |
| **Unite** | `A -> B` | `A -> (toutes les regles de B)` | `p(A->B) * p(B->...)` |

La **provenance** de chaque règle CNF est enregistree : c'est ce qui permettra, en section 5, de replier un arbre CNF vers l'arbre de la grammaire d'origine sans perdre aucune information.

**Périmètre du convertisseur** : les règles epsilon, cycles unitaires et collisions de plusieurs chemins vers une même règle CNF sont rejetés explicitement ; traiter ces collisions exige de distinguer les sommes inside des maxima Viterbi. Les exemples ci-dessous sont dans le domaine admis.


In [2]:
(List<Rule> Rules,Dictionary<Rule,double> Weights,Dictionary<Rule,Origin> Origins) Convertir(List<Rule> input,Dictionary<Rule,double> probabilities=null) {
    if(input.Distinct().Count()!=input.Count || input.Any(r=>r.Parts.Length==0))throw new ArgumentException("Règles dupliquées ou epsilon non prises en charge");
    var nt=input.Select(r=>r.Lhs).ToHashSet();var used=input.SelectMany(r=>r.Parts).Concat(nt).ToHashSet();var origin=new Dictionary<Rule,Origin>();var weights=new Dictionary<Rule,double>();var stage1=new List<Rule>();var terminals=new Dictionary<string,string>();
    string Fresh(string basis){string result=basis;int j=1;while(used.Contains(result))result=$"{basis}_{j++}";used.Add(result);return result;}
    foreach(var r in input){var parts=r.Parts;var transformed=new List<string>();foreach(string x in parts){if(parts.Length>=2 && !nt.Contains(x)){if(!terminals.ContainsKey(x)){string marker=Fresh("TERM_"+x);terminals[x]=marker;var lex=new Rule(marker,x);stage1.Add(lex);origin[lex]=new Origin("terminal",x,null);weights[lex]=1;}transformed.Add(terminals[x]);}else transformed.Add(x);}
        var q=new Rule(r.Lhs,string.Join(" ",transformed));if(weights.ContainsKey(q))throw new ArgumentException("Collision transformée");stage1.Add(q);origin[q]=new Origin("identity",null,null);weights[q]=probabilities?.GetValueOrDefault(r,1)??1;}
    var stage2=new List<Rule>();var w2=new Dictionary<Rule,double>();int ordinal=0;
    foreach(var r in stage1){var parts=r.Parts;if(parts.Length<=2){stage2.Add(r);w2[r]=weights[r];ordinal++;continue;}string parent=r.Lhs;bool first=true;int j=0;
      while(parts.Length-j>2){string marker=Fresh($"{r.Lhs}_bin_{ordinal}");var q=new Rule(parent,parts[j]+" "+marker);stage2.Add(q);origin[q]=new Origin(first?"head":"tail",null,null);w2[q]=first?weights[r]:1;first=false;parent=marker;j++;}
      var last=new Rule(parent,string.Join(" ",parts.Skip(j)));stage2.Add(last);origin[last]=new Origin("tail",null,null);w2[last]=1;ordinal++;}
    var by=stage2.GroupBy(r=>r.Lhs).ToDictionary(g=>g.Key,g=>g.ToList());var final=new List<Rule>();var wf=new Dictionary<Rule,double>();
    void Expand(string root,string current,List<Rule> chain,double factor,HashSet<string> path){if(path.Contains(current))throw new ArgumentException("Cycle unitaire non pris en charge");var nextPath=path.Append(current).ToHashSet();
      if(!by.ContainsKey(current))return;foreach(var r in by[current]){var parts=r.Parts;if(parts.Length==1 && nt.Contains(parts[0]))Expand(root,parts[0],chain.Append(r).ToList(),factor*w2[r],nextPath);else{var q=new Rule(root,r.Rhs);if(wf.ContainsKey(q))throw new ArgumentException("Collision de dérivations unitaires non prise en charge");final.Add(q);wf[q]=factor*w2[r];if(chain.Count>0)origin[q]=new Origin("unit",null,chain.Append(r).ToArray());}}}
    foreach(string lhs in by.Keys)Expand(lhs,lhs,new List<Rule>(),1,new HashSet<string>());
    return (final,wf,origin);
}
var structuralConversion=Convertir(productions);
var cnfStructural=structuralConversion.Rules;
Console.WriteLine($"{productions.Count} productions → {cnfStructural.Count} règles CNF ; aucune epsilon, cycle unitaire ou collision n'est admis");
foreach(var r in cnfStructural.Where(r=>!productions.Contains(r)))Console.WriteLine($"{r.Lhs} → {r.Rhs}");

25 productions → 31 règles CNF ; aucune epsilon, cycle unitaire ou collision n'est admis


NP → Det NP_bin_2


NP_bin_2 → N PP


VP → voit


VP → mange


VP → trouve


VP → coupe


VP → dort


VP → V VP_bin_5


VP_bin_5 → NP PP



warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotNet.Interactive.Formatting' matches identity 'Microsoft.AspNetCore.Html.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' of 'Microsoft.AspNetCore.Html.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotNet.Interactive.Formatting' matches identity 'Microsoft.AspNetCore.Html.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' of 'Microsoft.AspNetCore.Html.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotN

### Lecture — 31 règles CNF : neuf règles nouvelles, structurelles pour l'instant

Les 25 productions deviennent **31 règles CNF** : la verification `est_cnf` confirme que chaque règle est bien `A -> B C` ou `A -> w`. La conversion a introduit neuf règles nouvelles :

- **deux marqueurs de binarisation** : `NP_bin_2 -> N PP` (suite de `NP -> Det N PP`) et `VP_bin_5 -> NP PP` (suite de `VP -> V NP PP`), avec leurs tetes `NP -> Det NP_bin_2` et `VP -> V VP_bin_5` ;
- **cinq règles issues de l'unite** `VP -> V` : le verbe intransitif « remonte » comme production directe de `VP` (`VP -> voit`, `VP -> mange`, ...), ce qui permettra a un `V` seul d'etre immediatement un `VP` ;
- aucun preterminal `TERM_x` : notre grammaire ne place jamais un mot dans un rhs long, la transformation 1 reste disponible mais inutilisee ici.

**Point cle** : la conversion est *structurale* pour l'instant (toutes les probabilites valent 1 par defaut). Nous la referons en section 4 avec les probabilites estimees, et nous **verifierons alors explicitement** que la distribution est preservee.


## 3. Le recognizer CYK — programmation dynamique en table triangulaire

CYK (Cocke-Younger-Kasami) repond a la question d'appartenance : « la phrase est-elle dans le langage de la grammaire ? ». L'idee est la **programmation dynamique sur les sous-chaînes** : on remplit une table triangulaire `chart[i][l]` = ensemble des non-terminaux qui derivent les mots de la position `i` a `i+l`.

- **Base** (`l = 1`) : un mot `w` est derive par tous les non-terminaux qui ont la règle lexicale `A -> w` ;
- **Recurrence** (`l >= 2`) : pour chaque **point de coupure** `k`, si `B` derive `phrase[i:i+k]` et `C` derive `phrase[i+k:i+l]`, alors tout `A -> B C` derive `phrase[i:i+l]` ;
- **Verdict** : la phrase est acceptee si l'axiome `S` est dans `chart[0][n]` (la cellule au sommet, qui couvre toute la phrase).

La complexite est `O(n^3 * |G|)` — cubique dans la longueur de la phrase, lineaire dans la taille de la grammaire — et c'est **exactement** pour cette forme de recurrence a deux coupures que la CNF est necessaire.

On l'execute sur cinq phrases : deux **positives** (`P1` intransitive, `P2` transitive), une **ambigue** (`AMB`, huit mots, qui donnera deux analyses en section 5), et deux **negatives** pour des raisons différentes — `N1` s'arrete sur un determinant seul : le dernier mot `le` ouvre un groupe nominal que la phrase ne complete pas et `N2` contient un mot **hors lexique**.


In [3]:
string[] Words(string phrase)=>phrase.Split((char[])null,StringSplitOptions.RemoveEmptyEntries);
HashSet<string>[,] Table(List<Rule> grammar,string phrase){var words=Words(phrase);int n=words.Length;var table=new HashSet<string>[n,n+1];for(int i=0;i<n;i++)for(int len=0;len<=n;len++)table[i,len]=new HashSet<string>();
 for(int i=0;i<n;i++)foreach(var r in grammar)if(r.Parts.Length==1 && r.Rhs==words[i])table[i,1].Add(r.Lhs);
 for(int len=2;len<=n;len++)for(int i=0;i+len<=n;i++)for(int k=1;k<len;k++)foreach(var r in grammar){var p=r.Parts;if(p.Length==2 && table[i,k].Contains(p[0]) && table[i+k,len-k].Contains(p[1]))table[i,len].Add(r.Lhs);}return table;}
bool Reconnait(List<Rule> grammar,string phrase){var words=Words(phrase);return words.Length>0 && Table(grammar,phrase)[0,words.Length].Contains("S");}
var phrases=new Dictionary<string,string>{{"P1","le robot dort"},{"P2","le robot voit le ballon"},{"AMB","le robot voit le ballon dans la boite"},{"N1","le robot voit le"},{"N2","le zorglub voit le ballon"}};
foreach(var (name,phrase) in phrases)Console.WriteLine($"{name,-3} {phrase,-42} → {(Reconnait(cnfStructural,phrase)?"ACCEPTÉE":"REJETÉE")}");
if(Reconnait(cnfStructural,""))throw new Exception("Séquence vide acceptée sans epsilon");Console.WriteLine("Séquence vide : rejetée");

P1  le robot dort                              → ACCEPTÉE


P2  le robot voit le ballon                    → ACCEPTÉE


AMB le robot voit le ballon dans la boite      → ACCEPTÉE


N1  le robot voit le                           → REJETÉE


N2  le zorglub voit le ballon                  → REJETÉE


Séquence vide : rejetée


### Lecture — les verdicts CYK : trois acceptées, deux rejetées

Les verdicts sont ceux attendus : `P1`, `P2` et `AMB` sont **acceptees**, `N1` et `N2` sont **rejetees**. Remarquez que le recognizer ne distingue pas encore les deux negations : les deux se soldent par une cellule sommet vide. La section suivante ouvre la table pour *comprendre* les rejets, puis visualise la phrase ambigue.


### Ouvrir la table : la phrase ambigue

La table triangulaire de `AMB` (« le robot voit le ballon dans la boite », huit mots) se lit **de bas en haut** : la ligne `l=1` sont les mots, chaque ligne montante couvre des fragments de plus en plus longs, et la cellule sommet `l=8` — qui contient `S` — certifie l'acceptation. Deux representations : le `DataFrame` textuel, puis la grille colorée.


In [4]:
var ambWords=Words(phrases["AMB"]);int N=ambWords.Length;var chart=Table(cnfStructural,phrases["AMB"]);
for(int len=N;len>=1;len--){Console.Write($"l={len} ");for(int i=0;i+len<=N;i++)Console.Write($"[{i}:{string.Join(",",chart[i,len].OrderBy(x=>x))}] ");Console.WriteLine();}
var chartSvg=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='980' height='520' viewBox='0 0 980 520'><rect width='980' height='520' fill='white'/><text x='490' y='25' text-anchor='middle' font-size='16'>Table CYK triangulaire</text>");double cw=100,ch=48,left=100,top=55;
for(int len=1;len<=N;len++)for(int i=0;i+len<=N;i++){double x=left+i*cw,y=top+(N-len)*ch;string text=string.Join(",",chart[i,len].OrderBy(x=>x));chartSvg.Append($"<rect x='{F(x)}' y='{F(y)}' width='{F(cw-3)}' height='{F(ch-3)}' fill='{(text.Length>0?"#cfe3f7":"#f2f2f2")}' stroke='#777'/><text x='{F(x+48)}' y='{F(y+27)}' text-anchor='middle' font-size='11'>{E(text)}</text>");}
for(int i=0;i<N;i++)chartSvg.Append($"<text x='{F(left+i*cw+48)}' y='462' text-anchor='middle' font-size='12'>{E(ambWords[i])}</text>");
for(int len=1;len<=N;len++)chartSvg.Append($"<text x='87' y='{F(top+(N-len)*ch+27)}' text-anchor='end' font-size='12'>l={len}</text>");chartSvg.Append("<text x='490' y='500' text-anchor='middle' font-size='12'>position et mot de départ</text></svg>");display(new PlotSvg(chartSvg.ToString()));

l=8 

[0:S] 

l=7 

[0:] 

[1:] 

l=6 

... (35 outputs left out)

[5:P] 

[6:Det] 

[7:N] 

Table CYK triangulaire Det N V,VP Det N P Det N NP NP NP S VP PP NP_bin_2 S NP,VP_bin_5 VP S le robot voit le ballon dans la boite l=1 l=2 l=3 l=4 l=5 l=6 l=7 l=8 position et mot de départ

### Lecture — la table de la phrase ambiguë : trois cellules portent le sens

La grille montre la dynamique de bas en haut. Trois cellules portent le sens de l'ambiguïté :

- la cellule `(0, 8)` — toute la phrase — contient `S` : la phrase est bien dans le langage ;
- la cellule `(3, 5)` — « le ballon dans la boite » — contient **a la fois** `NP` *et* `VP_bin_5` : le même fragment de cinq mots peut etre un groupe nominal complet (attachement **nominal**) ou la suite binarisee d'un groupe verbal (attachement **verbal**) — c'est ici, materiellement, que vivent les deux lectures ;
- la cellule `(2, 6)` — « voit le ballon dans la boite » — contient `VP` : le groupe verbal transitif+complement est bien forme.

**Point cle** : CYK *reconnait* ; il ne dit pas encore **combien** d'analyses existent ni laquelle est preferable. C'est l'affaire des probabilites — sections 4 et 5.


### Pourquoi les phrases negatives sont-elles rejetees ?

Un « non » de CYK est muet ; la table, elle, se souvient de *ou* ca a coince. On inspecte deux indices : les mots sans catégorie a la ligne `l=1`, et le contenu de la cellule sommet.


In [5]:
foreach(string name in new[]{"N1","N2"}){var words=Words(phrases[name]);var t=Table(cnfStructural,phrases[name]);var oov=words.Where((w,i)=>t[i,1].Count==0).ToArray();Console.WriteLine($"{name} : {phrases[name]} | hors lexique : {(oov.Length==0?"aucun":string.Join(", ",oov))} | cellule finale : {string.Join(", ",t[0,words.Length])} | rejetée");}

N1 : le robot voit le | hors lexique : aucun | cellule finale :  | rejetée


N2 : le zorglub voit le ballon | hors lexique : zorglub | cellule finale :  | rejetée


### Lecture — deux rejets, deux causes : structurelle vs lexicale

Les deux rejets ont des causes distinctes, visibles dans la table :

| Cas | Indice dans la table | Cause |
|---|---|---|
| `N1` « le robot voit le » | aucun mot hors lexique, cellule sommet vide | **structurelle** : le determinant final `le` ouvre un groupe nominal qui reste inacheve — aucun `NP` ne couvre la fin de phrase, donc la cellule sommet reste vide |
| `N2` « le zorglub voit le ballon » | `zorglub` n'a **aucune** catégorie a `l=1` | **lexicale** : le mot est hors vocabulaire, toute derivation est bloquee des la base |

**Point cle** : ces deux echecs reviendront en section 7 — NLTK rejette `N2` par une **exception** documentee, la ou notre recognizer repond simplement « non » : deux conventions d'API pour le même fait linguistique.


## 4. La PCFG : estimer les probabilites sur un mini-treebank

Une **PCFG** est une CFG dont chaque règle `A -> alpha` porte une probabilite, avec pour chaque non-terminal `A` : la somme sur toutes ses règles vaut 1. La probabilite d'une **derivation** est le produit des probabilites des règles qu'elle utilise ; la probabilite d'une **phrase** est la somme sur toutes ses derivations.

D'ou viennent ces nombres ? D'un **treebank** : un corpus de phrases **déjà analysees** (des arbres), dont on compte les règles. L'estimation est le **maximum de vraisemblance** : `P(A -> alpha) = #(A -> alpha) / #(A)`.

Notre treebank est **embarque** : six arbres ecrits a la main dans le notebook (notation parenthesee de type Penn Treebank), tous dans le mini-francais de la section 1. Il est volontairement petit — chaque comptage reste verifiable a l'oeil — et volontairement **biaise** : cinq phrases transitives contre une intransitive, un seul attachement nominal. Ce biais n'est pas un defaut : il *pilote* la desambiguïsation de la section 5, et nous le mesurerons.


In [6]:
string[] treebank={
    "(S (NP (Det le) (N robot)) (VP (V voit) (NP (Det le) (N ballon))))",
    "(S (NP (Det le) (N robot)) (VP (V mange) (NP (Det le) (N camembert))))",
    "(S (NP (Det un) (N robot)) (VP (V trouve) (NP (Det un) (N ballon)) (PP (P dans) (NP (Det la) (N boite)))))",
    "(S (NP (Det la) (N boite)) (VP (V coupe) (NP (Det le) (N camembert)) (PP (P avec) (NP (Det le) (N couteau)))))",
    "(S (NP (Det le) (N robot)) (VP (V voit) (NP (Det le) (N ballon) (PP (P sur) (NP (Det la) (N table))))))",
    "(S (NP (Det le) (N robot)) (VP (V dort)))"
};
Node ParseTree(string text){var tokens=Regex.Matches(text,@"\(|\)|[^\s()]+").Select(x=>x.Value).ToArray();int pos=0;Node Read(){if(tokens[pos++]!="(")throw new ArgumentException("Arbre mal formé");string label=tokens[pos++];var children=new List<Node>();while(tokens[pos]!=")"){if(tokens[pos]=="(")children.Add(Read());else children.Add(new Node(tokens[pos++]));}pos++;return new Node(label,children:children);}var root=Read();if(pos!=tokens.Length)throw new ArgumentException("Arbre mal formé");return root;}
void CountTree(Node node,Dictionary<Rule,int> counts){if(node.Children.Count==0)return;var r=new Rule(node.Sym,string.Join(" ",node.Children.Select(x=>x.Sym)));counts[r]=counts.GetValueOrDefault(r)+1;foreach(var child in node.Children)CountTree(child,counts);}
Dictionary<Rule,int> CountAll(IEnumerable<string> trees){var counts=productions.ToDictionary(r=>r,_=>0);foreach(string tree in trees)CountTree(ParseTree(tree),counts);return counts;}
Dictionary<Rule,double> Estimate(Dictionary<Rule,int> counts){var totals=productions.GroupBy(r=>r.Lhs).ToDictionary(g=>g.Key,g=>g.Sum(r=>counts[r]));return productions.ToDictionary(r=>r,r=>(double)counts[r]/totals[r.Lhs]);}
var counts=CountAll(treebank);var prob=Estimate(counts);
foreach(var r in productions.OrderBy(r=>r.Lhs).ThenBy(r=>r.Rhs))Console.WriteLine($"{r.Lhs,-5} → {r.Rhs,-12} compte={counts[r],2} P={prob[r]:F4}");
foreach(var g in productions.GroupBy(r=>r.Lhs))if(Math.Abs(g.Sum(r=>prob[r])-1)>1e-12)throw new Exception("Probabilités non normalisées");
Console.WriteLine($"{treebank.Length} arbres ; {counts.Count} productions ; probabilités normalisées par père");

Det   → la           compte= 3 P=0.2143


Det   → le           compte= 9 P=0.6429


Det   → un           compte= 2 P=0.1429


Det   → une          compte= 0 P=0.0000


N     → ballon       compte= 3 P=0.2143


N     → boite        compte= 2 P=0.1429


... (16 outputs left out)

VP    → V            compte= 1 P=0.1667


VP    → V NP         compte= 3 P=0.5000


VP    → V NP PP      compte= 2 P=0.3333


6 arbres ; 25 productions ; probabilités normalisées par père


### Lecture — les probabilités MLE : le biais d'attachement et le zéro fatal

Le tableau donne, pour chacune des 25 productions, ses occurrences dans le treebank et la probabilite MLE resultante. Les mesures qui comptent pour la suite :

| Règle | Compte | P | Consequence |
|---|---|---|---|
| `VP -> V NP` | 3/6 | **0,50** | le transitif simple domine |
| `VP -> V NP PP` | 2/6 | **0,33** | l'attachement au verbe est frequent |
| `NP -> Det N PP` | 1/14 | **0,07** | l'attachement au nom est rare |
| `Det -> une` | 0/14 | **0** | règle jamais vue : probabilite nulle |

Pour la phrase ambigue, les deux lectures ne diffèrent que par **deux** règles locales : l'attachement verbal paie `P(VP -> V NP PP) x P(NP -> Det N)` = (2/6)(13/14), l'attachement nominal paie `P(VP -> V NP) x P(NP -> Det N PP)` = (3/6)(1/14). Leur rapport est 26/3 ~ 8,7 : ce **produit de règles locales**, pas les comptes bruts, est ce qui tranchera la phrase ambigue en section 5.

**Point cle** — la **faiblesse du MLE nu** : `Det -> une` recoit une probabilite **nulle**. Toute phrase future contenant « une » deviendra *intrinsèquement* impossible pour cette PCFG, même si la grammaire structurelle l'accepte : un seul zero dans un produit annule tout. C'est le problème classique du **sparse data**, que le lissage (add-one, interpolation...) corrige en production — volontairement hors scope ici, mais les exercices l'explorent.


### La CNF ponderee : la conversion preserve la distribution

On re-execute `Convertir`, cette fois avec les probabilites MLE. La conversion doit **preserver la normalisation** : pour chaque non-terminal, la somme des probabilites de ses règles CNF doit rester 1 — la tete binarisee herite de la probabilite d'origine, la suite porte 1, et une unite `A -> B` propage `p(A->B) * p(B -> ...)` le long de la chaîne.


In [7]:
var (cnf,pcnf,provenance)=Convertir(productions,prob);
foreach(var g in cnf.GroupBy(r=>r.Lhs)){double sum=g.Sum(r=>pcnf[r]);Console.WriteLine($"P(.|{g.Key})={sum:F6}");if(Math.Abs(sum-1)>1e-9)throw new Exception("Distribution non normalisée");}
Console.WriteLine($"CNF pondérée : {cnf.Count} règles ; normalisation locale vérifiée");

P(.|S)=1.000000


P(.|NP)=1.000000


P(.|NP_bin_2)=1.000000


P(.|VP)=1.000000


P(.|VP_bin_5)=1.000000


P(.|PP)=1.000000


P(.|Det)=1.000000


P(.|N)=1.000000


P(.|V)=1.000000


P(.|P)=1.000000


CNF pondérée : 31 règles ; normalisation locale vérifiée


### Lecture — la CNF pondérée : la distribution est préservée

Les 31 règles CNF sont desormais ponderees, et la verification affiche une somme de **1,000000** pour chaque non-terminal — y compris les marqueurs créés par la conversion, qui portent une seule règle de probabilité 1. La normalisation locale seule ne prouve pas la préservation des arbres. Les produits directs et les témoins NLTK contrôleront les dérivations des exemples admis.


## 5. Viterbi et inside : la meilleure derivation n'est pas la phrase

Le CYK probabiliste generalise la table triangulaire en y maintenant **deux** quantites par non-terminal et par fragment :

- **Viterbi** (`best[i][l][A]`) : probabilite de la *meilleure* derivation de ce fragment en `A` — un **maximum** sur les points de coupure et les règles ;
- **inside** (`inside[i][l][A]`) : probabilite *totale* de ce fragment en `A` — une **somme** sur toutes ses derivations.

Les deux recurrences se remplissent dans la même double boucle ; elles ne diffèrent que par l'opérateur d'agregation (`max` vs `+`). C'est la distinction centrale de ce notebook :

| Quantite | Opérateur | Sens |
|---|---|---|
| `P(meilleure derivation)` | max (Viterbi) | l'analyse la plus probable — *la reponse du parseur* |
| `P(phrase)` | somme (inside) | la vraisemblance totale de la phrase — *utile pour comparer des phrases entre elles* |

Confondre les deux est l'erreur classique : La meilleure dérivation ne peut jamais dépasser la somme inside. Une masse totale importante peut être répartie entre plusieurs dérivations ; le maximum ne décrit pas à lui seul cette répartition.

Pour retrouver *l'arbre* de la meilleure derivation, Viterbi memorise en plus un **backpointer** par cellule : le point de coupure `k` et la règle `A -> B C` qui ont produit le maximum. On le lit en remontant de la cellule sommet vers les feuilles.


In [8]:
class ChartResult {public string[] Words;public Dictionary<string,double>[,] Best,Inside;public Dictionary<string,(int Split,Rule Rule)>[,] Back;}
ChartResult Weighted(List<Rule> grammar,Dictionary<Rule,double> probabilities,string phrase){var words=Words(phrase);int n=words.Length;var best=new Dictionary<string,double>[n,n+1];var inside=new Dictionary<string,double>[n,n+1];var back=new Dictionary<string,(int,Rule)>[n,n+1];
 for(int i=0;i<n;i++)for(int len=0;len<=n;len++){best[i,len]=new();inside[i,len]=new();back[i,len]=new();}
 for(int i=0;i<n;i++)foreach(var r in grammar)if(r.Parts.Length==1 && r.Rhs==words[i] && probabilities[r]>0){best[i,1][r.Lhs]=probabilities[r];inside[i,1][r.Lhs]=probabilities[r];back[i,1][r.Lhs]=(0,r);}
 for(int len=2;len<=n;len++)for(int i=0;i+len<=n;i++)for(int k=1;k<len;k++)foreach(var r in grammar){var parts=r.Parts;if(parts.Length!=2 || !best[i,k].ContainsKey(parts[0]) || !best[i+k,len-k].ContainsKey(parts[1]))continue;double cand=best[i,k][parts[0]]*best[i+k,len-k][parts[1]]*probabilities[r];if(cand>best[i,len].GetValueOrDefault(r.Lhs)){best[i,len][r.Lhs]=cand;back[i,len][r.Lhs]=(k,r);}inside[i,len][r.Lhs]=inside[i,len].GetValueOrDefault(r.Lhs)+inside[i,k][parts[0]]*inside[i+k,len-k][parts[1]]*probabilities[r];}
 return new ChartResult{Words=words,Best=best,Inside=inside,Back=back};}
double Best(ChartResult r)=>r.Words.Length==0?0:r.Best[0,r.Words.Length].GetValueOrDefault("S");
double Inside(ChartResult r)=>r.Words.Length==0?0:r.Inside[0,r.Words.Length].GetValueOrDefault("S");
var weighted=Weighted(cnf,pcnf,phrases["AMB"]);double bestAmb=Best(weighted),insideAmb=Inside(weighted);
if(Best(Weighted(cnf,pcnf,""))!=0 || Inside(Weighted(cnf,pcnf,""))!=0 || bestAmb>insideAmb)throw new Exception("Contrat inside/Viterbi");
Console.WriteLine($"P(best)={bestAmb:E8} | P(phrase)={insideAmb:E8} | rapport={insideAmb/bestAmb:F4}; vide : 0/0");

P(best)=2.87105492E-005 | P(phrase)=3.20233049E-005 | rapport=1.1154; vide : 0/0


### Lecture — Viterbi contre inside : le rapport de 1,12 indique la part de masse hors de la meilleure dérivation

Sur la phrase ambigue :

- **P(meilleure derivation) = 2,87e-05** (Viterbi) ;
- **P(phrase) = 3,20e-05** (inside, toutes les derivations) ;
- **rapport = 1,12** : la meilleure derivation porte environ 90 % de la masse totale.

Ce rapport de 1,12 — proche de 1 mais distinct — est la signature d'une phrase **legerement** ambigue : une lecture domine, l'autre survit en minoritaire. Le parseur reconstruit alors l'arbre de la lecture dominante ; la section « ambiguite mesuree » comptera ensuite *toutes* les derivations.


### Reconstruire l'arbre : des backpointers CNF a la grammaire d'origine

Les backpointers decrivent un arbre **CNF** — plein de marqueurs les marqueurs créés et d'unites inlined qui n'existent pas dans la grammaire d'origine. La reconstruction replie ces artefacts grace a la **provenance** enregistree par le convertisseur : un marqueur de binarisation fait *remonter* ses enfants dans le parent (la règle `NP -> Det N PP` avait ete coupee en deux), et une unite `VP -> V -> voit` est *re-imbriquee* en `VP -> V`, `V -> voit`.


In [9]:
Node Reconstruct(ChartResult chart,int i,int len,string symbol){if(len<=0 || i>=chart.Words.Length || !chart.Back[i,len].ContainsKey(symbol))return null;var (k,r)=chart.Back[i,len][symbol];if(len==1)return new Node(symbol,r,chart.Words[i]);var p=r.Parts;return new Node(symbol,r,children:new[]{Reconstruct(chart,i,k,p[0]),Reconstruct(chart,i+k,len-k,p[1])});}
List<Node> Fold(Node node,Dictionary<Rule,Origin> origins){if(node==null)return new();origins.TryGetValue(node.Rule,out var o);
 if(o!=null && o.Kind=="unit"){var last=o.Chain[^1];var inner=new Node(last.Lhs,last,node.Word,node.Children);var folded=Fold(inner,origins);if(folded.Count!=1)throw new Exception("Unité vers marqueur non prise en charge");Node current=folded[0];for(int i=o.Chain.Length-2;i>=0;i--)current=new Node(o.Chain[i].Lhs,o.Chain[i],children:new[]{current});return new(){current};}
 if(o!=null && o.Kind=="terminal")return new(){new Node(o.Terminal)};
 var children=node.Children.SelectMany(c=>Fold(c,origins)).ToList();if(o!=null && o.Kind=="tail")return children;return new(){new Node(node.Sym,node.Rule,node.Word,children)};}
void PrintTree(Node node,string indent=""){Console.WriteLine(indent+node.Sym+(node.Word==null?"":" → "+node.Word));foreach(var child in node.Children)PrintTree(child,indent+"  ");}
var cnfTree=Reconstruct(weighted,0,N,"S");var arbre=Fold(cnfTree,provenance).Single();Console.WriteLine("Meilleure dérivation originale :");PrintTree(arbre);

Meilleure dérivation originale :


S


  NP


    Det → le


    N → robot


  VP


... (5 outputs left out)

      P → dans


      NP


        Det → la


        N → boite


### Lecture — l'arbre reconstruit : la lecture verbale, dans la grammaire d'origine

L'arbre reconstruit est dans la grammaire d'origine — plus aucun marqueur de binarisation — et il realise la lecture **verbale** : le complement `dans la boite` est enfant direct du `VP`, au même niveau que le groupe nominal `le ballon`. Sous cette lecture, c'est l'action de voir qui se deroule « dans la boite ». La lecture alternative (le *ballon* est dans la boite) existe aussi : elle sera montree et quantifiee dans deux cellules.

La cellule suivante fait le contrôle de coherence decisif : **recalculer la probabilite de cet arbre original** en multipliant directement les règles de la grammaire ponderee, et verifier qu'on retombe sur le score Viterbi calcule sur la CNF.


In [10]:
double Product(Node node,Dictionary<Rule,double> weights){if(node.Word==null && node.Children.Count==0)return 1;var rule=node.Word!=null?new Rule(node.Sym,node.Word):new Rule(node.Sym,string.Join(" ",node.Children.Select(c=>c.Sym)));return weights[rule]*node.Children.Aggregate(1.0,(p,c)=>p*Product(c,weights));}
double product=Product(arbre,prob);var p1chart=Weighted(cnf,pcnf,phrases["P1"]);var p1=Fold(Reconstruct(p1chart,0,p1chart.Words.Length,"S"),provenance).Single();
if(Math.Abs(product-bestAmb)>1e-12 || Math.Abs(Product(p1,prob)-Best(p1chart))>1e-12 || Reconstruct(Weighted(cnf,pcnf,""),0,0,"S")!=null)throw new Exception("Provenance incorrecte");
Console.WriteLine($"Produit direct AMB={product:E8}, Viterbi={bestAmb:E8} ; P1={Product(p1,prob):E8} ; vide sans arbre");
// Oracles sur chaîne unitaire, deux ternaires distinctes et terminal mixte.
var chain=new List<Rule>{new("S","A"),new("A","B"),new("B","b")};if(!Reconnait(Convertir(chain).Rules,"b"))throw new Exception("Chaîne unitaire");
var alts=new List<Rule>{new("S","A B C"),new("S","A B D"),new("A","a"),new("B","b"),new("C","c"),new("D","d")};var aw=alts.ToDictionary(r=>r,r=>r.Rhs=="A B C"?0.3:r.Rhs=="A B D"?0.7:1.0);var (ac,ap,ao)=Convertir(alts,aw);if(Math.Abs(Inside(Weighted(ac,ap,"a b c"))-0.3)>1e-12)throw new Exception("Marqueurs ternaires");
var mix=new List<Rule>{new("S","a B C"),new("S","z"),new("B","b"),new("C","c")};var mw=mix.ToDictionary(r=>r,r=>r.Rhs=="a B C"?0.3:r.Lhs=="S"?0.7:1.0);var (mc,mp,mo)=Convertir(mix,mw);var mr=Weighted(mc,mp,"a b c");var mt=Fold(Reconstruct(mr,0,3,"S"),mo).Single();if(Math.Abs(Inside(mr)-0.3)>1e-12 || Math.Abs(Product(mt,mw)-0.3)>1e-12)throw new Exception("Terminal mixte");
foreach(var bad in new[]{new List<Rule>{new("S","")},new List<Rule>{new("S","A"),new("A","S")},new List<Rule>{new("S","A"),new("S","B"),new("A","a"),new("B","a")}}){bool rejected=false;try{Convertir(bad);}catch(ArgumentException){rejected=true;}if(!rejected)throw new Exception("Grammaire hors domaine acceptée");}
Console.WriteLine("Chaîne unitaire, marqueurs, terminal mixte et rejets : fixtures validées");

Produit direct AMB=2.87105492E-005, Viterbi=2.87105492E-005 ; P1=5.92201166E-003 ; vide sans arbre


Chaîne unitaire, marqueurs, terminal mixte et rejets : fixtures validées


### Lecture — le contrôle de cohérence : produit direct = score Viterbi

Le produit direct sur l'arbre reconstruit vaut **2,871055e-05**, identique au score Viterbi calcule sur la CNF : conversion ponderee, backpointers et repli de l'arbre sont mutuellement coherents. Ce contrôle vérifie la préservation sur cet arbre ; P1 et les petits contre-exemples contrôlent également les unités et les terminaux isolés. Il ne prouve pas le comportement pour toute grammaire.

> **Note technique** : sur de vraies grammaires, les produits de probabilites deviennent si petits qu'on travaille en **log-probabilites** (sommes au lieu de produits) pour eviter l'underflow flottant. Sur huit mots, les probabilites directes restent lisibles ; c'est un luxe pedagogique.


## 6. Ambiguïté mesuree : compter les derivations et peser les lectures

Viterbi donne la *meilleure* ; pour **mesurer** l'ambiguite il faut les *enumerer toutes*. On modifie la recurrence en une enumeration memoisee : pour chaque fragment et chaque non-terminal, tous les arbres possibles, puis on replie chacun dans la grammaire d'origine et on calcule sa probabilite. Deux mesures resultent :

- le **nombre de derivations candidates** (le degré d'ambiguite brute) ;
- la **probabilite de chaque lecture**, comparee a la probabilite totale de la phrase.

Le contrôle de coherence : la somme des probabilites de toutes les derivations doit egaler le score `inside` de la section 5.


In [11]:
List<Node> Enumerer(List<Rule> grammar,Dictionary<Rule,double> weights,string[] words,int i,int len,string symbol,Dictionary<(int,int,string),List<Node>> memo=null){memo??=new();var key=(i,len,symbol);if(memo.ContainsKey(key))return memo[key];var result=new List<Node>();if(len<=0)return result;
 if(len==1){foreach(var r in grammar)if(r.Lhs==symbol && r.Parts.Length==1 && r.Rhs==words[i])result.Add(new Node(symbol,r,words[i]));}
 else for(int k=1;k<len;k++)foreach(var r in grammar){var p=r.Parts;if(r.Lhs!=symbol || p.Length!=2)continue;foreach(var a in Enumerer(grammar,weights,words,i,k,p[0],memo))foreach(var b in Enumerer(grammar,weights,words,i+k,len-k,p[1],memo))result.Add(new Node(symbol,r,children:new[]{a,b}));}
 memo[key]=result;return result;}
bool IsNominal(Node node)=>node.Sym=="NP" && node.Children.Count==3 || node.Children.Any(IsNominal);
var derivations=Enumerer(cnf,pcnf,ambWords,0,N,"S").Select(n=>Fold(n,provenance).Single()).Select(n=>(Tree:n,Weight:Product(n,prob))).OrderByDescending(x=>x.Weight).ToArray();
foreach(var (item,index) in derivations.Select((x,i)=>(x,i)))Console.WriteLine($"{index+1} {(IsNominal(item.Tree)?"NOM":"VERBE"),-6} P={item.Weight:E8} part={item.Weight/insideAmb:P2}");
if(Math.Abs(derivations.Sum(x=>x.Weight)-insideAmb)>1e-12)throw new Exception("Somme inside incorrecte");Console.WriteLine($"{derivations.Length} dérivations ; somme={derivations.Sum(x=>x.Weight):E8} = inside {insideAmb:E8}");

1 VERBE  P=2.87105492E-005 part=89.66 %


2 NOM    P=3.31275568E-006 part=10.34 %


2 dérivations ; somme=3.20233049E-005 = inside 3.20233049E-005


Les probabilités énumérées somment à la masse inside. Visualisons les deux lectures sur une échelle logarithmique.

In [12]:
var bars=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='380' viewBox='0 0 760 380'><rect width='760' height='380' fill='white'/><text x='380' y='25' text-anchor='middle' font-size='16'>Deux lectures : probabilités</text>");double leftB=140,topB=60,widthB=550;for(int j=0;j<3;j++){double x=leftB+j*widthB/2;bars.Append($"<line x1='{F(x)}' y1='65' x2='{F(x)}' y2='265' stroke='#ddd'/><text x='{F(x)}' y='285' text-anchor='middle' font-size='12'>1e{j-6}</text>");}
for(int j=0;j<derivations.Length;j++){double y=95+j*75, length=(Math.Log10(derivations[j].Weight)+6)/2*widthB;bars.Append($"<text x='130' y='{F(y+20)}' text-anchor='end' font-size='12'>{(IsNominal(derivations[j].Tree)?"nominal":"verbal")}</text><rect x='{F(leftB)}' y='{F(y)}' width='{F(length)}' height='30' fill='{(j==0?"#2e7d32":"#c62828")}'/><text x='{F(leftB+length+6)}' y='{F(y+20)}' font-size='12'>{E(derivations[j].Weight.ToString("E2",CultureInfo.InvariantCulture))} ({F(100*derivations[j].Weight/insideAmb)}%)</text>");}
bars.Append("<path d='M140 60 V270 H690' fill='none' stroke='black'/><text x='415' y='320' text-anchor='middle' font-size='12'>P(dérivation), échelle logarithmique</text><text x='140' y='350' font-size='12'>Barres : lectures verbale et nominale</text></svg>");display(new PlotSvg(bars.ToString()));Console.WriteLine("Figure des deux probabilités en échelle logarithmique.");

Deux lectures : probabilités 1e-6 1e-5 1e-4 verbal 2.87E-005 (89.655%) nominal 3.31E-006 (10.345%) P(dérivation), échelle logarithmique Barres : lectures verbale et nominale

Figure des deux probabilités en échelle logarithmique.


### Lecture — l'ambiguïté mesurée : deux dérivations, ~90 % contre ~10 %

La phrase a exactement **2 derivations**, et l'enumeration confirme le diagnostic du treebank :

| Rang | Lecture | P(derivation) | Part de P(phrase) |
|---|---|---|---|
| 1 | attachement au **verbe** | 2,87e-05 | **89,7 %** |
| 2 | attachement au **nom** | 3,31e-06 | 10,3 % |

La somme des deux egale exactement le score inside — le calcul est boucle. La lecture verbale domine d'un facteur ~8,7, ce qui reflete fidelement le treebank (attachements verbaux 2/6 contre nominal 1/14, combines aux probabilites des règles `V -> voit` et lexicales). L'histogramme ci-dessus (échelle logarithmique) visualise l'écart.

Le bar chart montre les deux masses de probabilite separees par presque un ordre de grandeur (echelle log). C'est une phrase **faiblement ambigue** pour cette PCFG : la masse se concentre a ~90 % sur une lecture. Sur une grammaire reelle, l'ambiguite croit fortement avec la taille du lexique et des règles : les lectures rivales s'y multiplient bien au-dela de nos deux cas — ce qui justifie l'existence même des PCFG : sans probabilites, un parseur ne saurait pas *trancher*.

La cellule suivante montre la derivation Viterbi en figure : c'est la reponse que le parseur « propose » a l'utilisateur.


In [13]:
var nodes=new List<(Node Node,int Depth,double X)>();int leaf=0;
double Layout(Node node,int depth){if(node.Children.Count==0){double x=70+leaf++*95;nodes.Add((node,depth,x));return x;}var xs=node.Children.Select(child=>Layout(child,depth+1)).ToArray();double center=xs.Average();nodes.Add((node,depth,center));return center;}
Layout(arbre,0);var positions=nodes.ToDictionary(x=>x.Node,x=>(x.Depth,x.X));var treeSvg=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='980' height='520' viewBox='0 0 980 520'><rect width='980' height='520' fill='white'/><text x='490' y='25' text-anchor='middle' font-size='16'>Dérivation Viterbi : attachement verbal</text>");
foreach(var item in nodes)foreach(var child in item.Node.Children){var pos=positions[child];treeSvg.Append($"<line x1='{F(item.X)}' y1='{F(60+item.Depth*60)}' x2='{F(pos.X)}' y2='{F(60+pos.Depth*60)}' stroke='#555'/>");}
foreach(var item in nodes){double y=60+item.Depth*60;string label=item.Node.Sym+(item.Node.Word==null?"":" : "+item.Node.Word);treeSvg.Append($"<rect x='{F(item.X-34)}' y='{F(y-14)}' width='68' height='28' rx='4' fill='{(item.Node.Word==null?"#cfe3f7":"#fff3cd")}' stroke='#456'/><text x='{F(item.X)}' y='{F(y+4)}' text-anchor='middle' font-size='11'>{E(label)}</text>");}
treeSvg.Append("<text x='70' y='495' font-size='12'>Bleu : constituant · jaune : mot terminal</text></svg>");display(new PlotSvg(treeSvg.ToString()));Console.WriteLine("Arbre reconstruit dans la grammaire originale.");

Dérivation Viterbi : attachement verbal Det : le N : robot NP V : voit Det : le N : ballon NP P : dans Det : la N : boite NP PP VP S Bleu : constituant · jaune : mot terminal

Arbre reconstruit dans la grammaire originale.


### Lecture — l'arbre dessiné : le PP est le troisième enfant de VP

L'arbre dessine confirme visuellement la lecture dominante : `PP (P dans, NP la boite)` est le **troisieme enfant de VP**, pas un complement du nom `ballon`. La figure est le format de sortie canonique d'un parseur : tout système TAL qui analyse une phrase produit une telle structure, que ce soit par PCFG, par transition-based parsing ou par un LLM structure.


### Exercice 1 — Compter les analyses

Retournez le nombre de dérivations pour chaque phrase, y compris les entrées vides et hors lexique. Utilisez l'énumération de la section 6 ; les fixtures exigent les clés exactes et des comptes variés.

In [14]:
// Exercice 1 : compter les analyses de chaque phrase.
Dictionary<string,int> CompteDerivations(Dictionary<string,string> inputs){ /* TODO : énumérer pour chaque entrée. */ return null; }
var ex1=CompteDerivations(new(){{"p1",phrases["P1"]},{"amb",phrases["AMB"]},{"vide",""},{"inconnu","zorglub"}});
if(ex1==null)Console.WriteLine("Exercice 1 à compléter");else {var autre=CompteDerivations(new(){{"p2",phrases["P2"]},{"n1",phrases["N1"]}});bool ok=ex1.Count==4 && ex1.GetValueOrDefault("p1",-1)==1 && ex1.GetValueOrDefault("amb",-1)==2 && ex1.GetValueOrDefault("vide",-1)==0 && ex1.GetValueOrDefault("inconnu",-1)==0 && autre!=null && autre.Count==2 && autre.GetValueOrDefault("p2",-1)==1 && autre.GetValueOrDefault("n1",-1)==0;Console.WriteLine(ok?"Fixtures de comptage validées":"Réponse incorrecte");}

Exercice 1 à compléter


## 7. Témoin NLTK : verifier notre parseur contre la reference

Une implementation from scratch n'est credible que si elle **coincide** avec l'outil de reference. NLTK fournit exactement nos trois algorithmes : `ChartParser` (toutes les analyses), `ViterbiParser` (la meilleure), `InsideChartParser` (toutes les analyses ponderees). On lui donne la **même** PCFG — construite production par production depuis nos probabilites MLE — et les **mêmes** phrases, puis on compare chiffre a chiffre, avec des assertions d'egalite (tolerance flottante 1e-9 relative).

Ce témoin fait deux choses : il valide notre code (toute divergence invaliderait une hypothese), et il **ancre** ce notebook dans l'ecosysteme reel — NLTK est la bibliotheque de reference du TAL pedagogique.


In [15]:
if(!PythonEngine.IsInitialized){var dll=Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");if(string.IsNullOrWhiteSpace(dll))throw new InvalidOperationException("Définir PYTHONNET_PYDLL avant Jupyter.");Runtime.PythonDLL=dll;PythonEngine.Initialize();PythonEngine.BeginAllowThreads();}
using(Py.GIL()){
 dynamic gr=Py.Import("nltk.grammar"),parsers=Py.Import("nltk.parse"),builtins=Py.Import("builtins"),nltk=Py.Import("nltk");
 var nt=productions.Select(r=>r.Lhs).ToHashSet();string grammarText=string.Join("\n",productions.Select(r=>$"{r.Lhs} -> {string.Join(" ",r.Parts.Select(x=>nt.Contains(x)?x:"'"+x+"'"))} [{prob[r].ToString("R",CultureInfo.InvariantCulture)}]"));
 dynamic grammar=gr.PCFG.fromstring(grammarText);dynamic parser=parsers.ChartParser(grammar),insideParser=parsers.InsideChartParser(grammar),viterbiParser=parsers.ViterbiParser(grammar);
 Console.WriteLine($"NLTK {nltk.__version__} : ChartParser / InsideChartParser / ViterbiParser");
 foreach(string name in new[]{"P1","P2","AMB"}) {var words=Words(phrases[name]);using var pyWords=new PyList();foreach(var word in words){using var w=new PyString(word);pyWords.Append(w);}using PyObject all=builtins.list(parser.parse(pyWords));using PyObject weightedTrees=builtins.list(insideParser.parse(pyWords));using PyObject bests=builtins.list(viterbiParser.parse(pyWords));double sum=0;for(int i=0;i<weightedTrees.Length();i++){using var t=weightedTrees.GetItem(i);sum+=(double)((dynamic)t).prob();}using var best=bests.GetItem(0);double bestP=(double)((dynamic)best).prob();var own=Weighted(cnf,pcnf,phrases[name]);if(Math.Abs(bestP-Best(own))>1e-9 || Math.Abs(sum-Inside(own))>1e-9)throw new Exception("Témoin divergent");Console.WriteLine($"{name} : {all.Length()} analyses | best NLTK={bestP:E8}, C#={Best(own):E8} | inside NLTK={sum:E8}, C#={Inside(own):E8}");if(name=="AMB")Console.WriteLine($"Arbre NLTK : {((dynamic)best).pformat()}");}
 using var unknown=new PyList();foreach(var word in Words(phrases["N2"])){using var w=new PyString(word);unknown.Append(w);}try{builtins.list(parser.parse(unknown));throw new Exception("OOV accepté");}catch(PythonException ex){if(ex.Type.Name!="ValueError")throw;Console.WriteLine("N2 hors lexique : NLTK ValueError ; CYK rejette");}
}

NLTK 3.10.4 : ChartParser / InsideChartParser / ViterbiParser


P1 : 1 analyses | best NLTK=5.92201166E-003, C#=5.92201166E-003 | inside NLTK=5.92201166E-003, C#=5.92201166E-003


P2 : 1 analyses | best NLTK=4.54510079E-003, C#=4.54510079E-003 | inside NLTK=4.54510079E-003, C#=4.54510079E-003


AMB : 2 analyses | best NLTK=2.87105492E-005, C#=2.87105492E-005 | inside NLTK=3.20233049E-005, C#=3.20233049E-005


Arbre NLTK : (S
  (NP (Det le) (N robot))
  (VP
    (V voit)
    (NP (Det le) (N ballon))
    (PP (P dans) (NP (Det la) (N boite)))))


N2 hors lexique : NLTK ValueError ; CYK rejette


### Lecture — la concordance NLTK : exacte sur les trois phrases

La concordance est **exacte** sur les trois phrases positives/ambigues et les deux quantites :

| Cas | #parses NLTK | P_best NLTK | P_best (nous) | P_inside NLTK | P_inside (nous) |
|---|---|---|---|---|---|
| P1 | 1 | 5,92e-03 | 5,92e-03 | 5,92e-03 | 5,92e-03 |
| P2 | 1 | 4,55e-03 | 4,55e-03 | 4,55e-03 | 4,55e-03 |
| AMB | **2** | 2,87e-05 | 2,87e-05 | 3,20e-05 | 3,20e-05 |

L'arbre Viterbi de NLTK, affiche en notation parenthesee, est celui de la lecture **verbale** — identique a notre reconstruction. Enfin, sur le cas negatif `N2`, NLTK leve `ValueError: Grammar does not cover some of the input words` : la ou notre table repond « non » par une cellule vide, NLTK **exige** que chaque mot soit couvert par le lexique et refuse de lancer le parsing — deux conventions d'API face a un mot inconnu.

**Point cle** : P1 et P2 ont une **seule** derivation — pour elles, Viterbi et inside coincident. Le rapport 1,12 indique ici la masse des autres dérivations positives relativement à la meilleure. Un rapport de 1 peut aussi coexister avec plusieurs dérivations dont une seule a un poids positif.


### Exercice 2 — Renforcer la lecture nominale

Ajoutez `copies` analyses nominales au treebank, réestimez les 25 probabilités, puis mesurez la part de masse verbale sur la phrase ambiguë. Comparez 0, 1, 4 et 5 copies ; les données d'origine restent inchangées.

In [16]:
// Exercice 2 : compléter le treebank avec copies de l'analyse nominale.
Dictionary<Rule,double> TreebankEtendu(int copies=1){ /* TODO : recompter les arbres et estimer les 25 règles. */ return null; }
double? PartVerbale(Dictionary<Rule,double> probabilities){ /* TODO : convertir, énumérer et calculer la part verbale. */ return null; }
var ex2=TreebankEtendu(1);
if(ex2==null)Console.WriteLine("Exercice 2 à compléter");else {bool ok=true;foreach(var (copies,expected) in new[]{(0,0.896551724137931),(1,0.789473684210526),(4,0.545454545454545),(5,0.489361702127660)}){var p=TreebankEtendu(copies);var share=PartVerbale(p);ok &= p!=null && p.Count==productions.Count && productions.All(r=>p.ContainsKey(r) && double.IsFinite(p[r]) && p[r]>=0 && p[r]<=1) && productions.GroupBy(r=>r.Lhs).All(g=>Math.Abs(g.Sum(r=>p[r])-1)<1e-9) && share.HasValue && double.IsFinite(share.Value) && Math.Abs(share.Value-expected)<1e-9;}Console.WriteLine(ok?"Fixtures extension validées":"Réponse incorrecte");}

Exercice 2 à compléter


## 8. Exercices — récapitulatif

Les exercices répartis demandent de [compter les analyses](#exercice-1--compter-les-analyses), [modifier le treebank](#exercice-2--renforcer-la-lecture-nominale), puis [lisser les règles non observées](#exercice-3--lisser-les-règles-non-observées). Ils conservent des stubs et des fixtures contrastées.

### Exercice 3 — Lisser les règles non observées

Appliquez add-one à **toutes les 25 productions**, y compris les comptes nuls, puis calculez inside pour « une robot voit le ballon ». `une` est dans le lexique mais absente du treebank ; `zorglub` reste hors lexique et sa probabilité est nulle sans extension lexicale.

In [17]:
// Exercice 3 : add-one sur toutes les productions et probabilité inside.
Dictionary<Rule,double> MleLisse(Dictionary<Rule,int> c,double alpha=1){ /* TODO : lisser les 25 règles et normaliser par père. */ return null; }
double? ProbaPhrase(Dictionary<Rule,double> p,string phrase){ /* TODO : convertir et calculer inside, OOV → 0. */ return null; }
var ex3=MleLisse(counts,1);
if(ex3==null)Console.WriteLine("Exercice 3 à compléter");else {var autre=MleLisse(counts,0.5);var pUne=ProbaPhrase(ex3,"une robot voit le ballon");var pOov=ProbaPhrase(ex3,"le zorglub voit le ballon");
 bool ok=autre!=null && ex3.Count==productions.Count && autre.Count==productions.Count && productions.All(r=>ex3.ContainsKey(r) && autre.ContainsKey(r) && double.IsFinite(ex3[r]) && double.IsFinite(autre[r]) && ex3[r]>=0 && ex3[r]<=1 && autre[r]>=0 && autre[r]<=1) && productions.GroupBy(r=>r.Lhs).All(g=>Math.Abs(g.Sum(r=>ex3[r])-1)<1e-9) && Math.Abs(ex3[new Rule("Det","une")]-1.0/18)<1e-9 && Math.Abs(autre[new Rule("Det","une")]-0.5/16)<1e-9 && pUne.HasValue && double.IsFinite(pUne.Value) && Math.Abs(pUne.Value-0.00017185746352413016)<1e-12 && pOov.HasValue && pOov.Value==0;
 Console.WriteLine(ok?"Fixtures add-one et OOV validées":"Réponse incorrecte");}

Exercice 3 à compléter


## À retenir

Ce notebook a construit, mesure et verifie une chaîne complete de parsing probabiliste sur un mini-francais fil rouge :

- une **CFG explicite** (7 règles syntaxiques, 18 lexicales) et sa conversion **CNF** documentee — 31 règles sur cette grammaire, normalisation locale et produits d’arbres contrôlés ;
- le **recognizer CYK** from scratch en table triangulaire, avec verdicts positifs, negatifs *expliques par la table* (echec structurel vs lexical), et visualisation cellule par cellule de la phrase ambigue ;
- une **PCFG estimee par MLE** sur un mini-treebank de 6 arbres embarques, avec le biais d'attachement mesure (verbal 0,33 vs nominal 0,07) et le defaut du MLE nu exhibe (`Det -> une` a 0) ;
- le decodage **Viterbi + backpointers** : meilleure derivation 2,87e-05, arbre reconstruit dans la grammaire d'origine, coherence produit-direct/Viterbi exacte ;
- l'**ambiguite mesuree** : 2 derivations, 89,7 % / 10,3 %, et la distinction P(meilleure) ≠ P(phrase) = 3,20e-05 demontree par l'egalite inside = somme des derivations ;
- le **temoin NLTK** : concordance exacte sur #parses, P_best et P_inside pour les trois cas, et comportement contrasté sur le mot inconnu (exception vs cellule vide).

**Points clés a retenir** :

1. La **CNF** est ici une representation exacte pour les grammaires admises : elle existe pour rendre la recurrence a deux coupures (et donc la complexite cubique) possible ;
2. **Viterbi** repond « quelle analyse ? », **inside** repond « quelle vraisemblance ? » — deux questions différentes, deux opérateurs (`max` vs somme) dans la même table ;
3. Les probabilites d'une PCFG **sont** le treebank : changer un arbre dans le corpus d'apprentissage deplace la lecture dominante (exercice 2) ;
4. Un parseur from scratch se **verifie** contre un temoin de reference ; l'egalite a 1e-9 est le critere.

**Ponts** : [`01_TAL_Du_Mot_Aux_Dependances.ipynb`](01_TAL_Du_Mot_Aux_Dependances.ipynb) pour l'etage mot/dependances que cette grammaire sous-tend ; la serie `Search` pour la programmation dynamique qui est le moteur de CYK ; [`RAG-et-Memoire-Semantique/04-Tokenisation-From-Scratch.ipynb`](../GenAI/RAG-et-Memoire-Semantique/04-Tokenisation-From-Scratch.ipynb) pour la frontiere entre analyse symbolique (ici) et encodage sous-mot des LLM. Pour aller plus loin : l'**algorithme Earley** (leve la contrainte CNF), le **PCFG lexicalise** (Collins), et les parseurs a transitions qui ont succede aux PCFG en production.


## Vérifiez votre compréhension

1. La CNF est-elle une approximation de la grammaire d'origine ? Que gagne-t-on à payer sa conversion ?
2. Viterbi et inside remplissent la même table CYK : quelle question chacun répond-il, et que mesure leur rapport de 1,12 sur la phrase ambiguë ?
3. D'où viennent les probabilités d'une PCFG — et que prouve l'exercice 2 en ajoutant un seul arbre au treebank ?
4. Que vaut `P(Det → une)` en MLE nu, et quel lissage lui rend une valeur non nulle ?

<details><summary>Réponses</summary>

1. **Sur le domaine admis**, la conversion conserve les dérivations et leurs poids ; la normalisation locale seule ne prouve pas cette propriété. Elle rend possible la récurrence à deux coupures, donc la complexité **cubique** de CYK.
2. Viterbi répond « **quelle analyse** » (max), inside répond « **quelle vraisemblance** » (somme). Le rapport P(phrase)/P(meilleure) ≈ 1,12 mesure l'**ambiguïté** — et vaut 1 lorsqu’une seule dérivation a un poids positif, même si d’autres dérivations de poids nul existent.
3. Elles **sont** le treebank (estimation MLE par comptage). Un seul arbre nominal ajouté fait chuter l'avantage verbal de 8,7 à 3,75 sans le renverser : les probabilités suivent le corpus.
4. **Zéro** — « une » n'apparaît dans aucun arbre du treebank. Le lissage **add-one** rend cette règle positive parmi les 25 productions connues ; il ne crée pas de règle pour un mot hors lexique.

</details>


## Pour aller plus loin

**Carnet suivant** : [05-HMM-Viterbi.ipynb](05-HMM-Viterbi.ipynb) — la même programmation dynamique, des scores cette fois appris : le treillis de Viterbi y étiquette une séquence au lieu d'analyser un arbre.

- **Lectures** : l'algorithme d'**Earley** (lève la contrainte CNF) ; Collins, « Head-Driven Statistical Models for Natural Language Parsing » (PCFG lexicalisée) ; Jurafsky & Martin, ch. « Constituency Parsing ».
- **Dans le dépôt** : [01_TAL_Du_Mot_Aux_Dependances.ipynb](01_TAL_Du_Mot_Aux_Dependances.ipynb) pour l'étage mots/dépendances que cette grammaire sous-tend ; la série `Search`, cœur de la programmation dynamique qui fait tourner CYK.
